In [ ]:
# Cell 1: clone repo
!git clone https://github.com/nnzhan/Graph-WaveNet.git
%cd Graph-WaveNet

In [ ]:
# Clone DCRNN để lấy file adj
!git clone https://github.com/liyaguang/DCRNN.git

# Kiểm tra file có ở đó không
!ls DCRNN/data/sensor_graph/

In [ ]:
# Copy file adj vào đúng chỗ Graph-WaveNet cần
!mkdir -p /kaggle/working/Graph-WaveNet/data/sensor_graph

# METR-LA
!cp DCRNN/data/sensor_graph/adj_mx.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx.pkl

# PEMS-BAY
!cp DCRNN/data/sensor_graph/adj_mx_bay.pkl \
    /kaggle/working/Graph-WaveNet/data/sensor_graph/adj_mx_bay.pkl

In [ ]:
# Kiểm tra lại
!ls /kaggle/working/Graph-WaveNet/data/sensor_graph/

In [ ]:
%%writefile /kaggle/working/Graph-WaveNet/model.py
"""
FULL MODEL for METR-LA
- Dynamic Adaptive Adjacency
- Local Causal Skip Attention  
- CausalWindowAttnTCN
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# Local Causal Skip Attention
# =============================================================================
class LocalCausalSkipAttention(nn.Module):
    def __init__(self, channels, num_heads=4, window_size=16, dropout=0.1):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.window_size = window_size
        self.scale = math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(channels, channels, bias=False)
        self.k_proj = nn.Linear(channels, channels, bias=False)
        self.v_proj = nn.Linear(channels, channels, bias=False)
        self.out_proj = nn.Linear(channels, channels)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        B, C, N, T = x.shape
        x_flat = x.permute(0, 2, 3, 1).reshape(B * N, T, C)

        Q = self.q_proj(x_flat)
        K = self.k_proj(x_flat)
        V = self.v_proj(x_flat)

        H, D = self.num_heads, self.head_dim
        Q = Q.view(B*N, T, H, D).transpose(1, 2)
        K = K.view(B*N, T, H, D).transpose(1, 2)
        V = V.view(B*N, T, H, D).transpose(1, 2)

        attn = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        mask = torch.ones(T, T, dtype=torch.bool, device=x.device)
        for i in range(T):
            lo = max(0, i - self.window_size + 1)
            mask[i, lo:i+1] = False
        mask = mask.unsqueeze(0).unsqueeze(0).expand(B*N, H, -1, -1)
        attn = attn.masked_fill(mask, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).reshape(B*N, T, C)
        out = self.out_proj(out)
        out = self.norm(out)

        return out.view(B, N, T, C).permute(0, 3, 1, 2)


# =============================================================================
# CausalWindowAttnTCN
# =============================================================================
def _causal_window_mask(seq_len: int, window: int, device) -> torch.Tensor:
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
    for i in range(seq_len):
        lo = max(0, i - window + 1)
        mask[i, lo : i + 1] = False
    return mask

class RelativePositionalEncoding(nn.Module):
    def __init__(self, num_heads: int, max_len: int = 64):
        super().__init__()
        self.num_heads = num_heads
        self.rel_bias = nn.Embedding(max_len, num_heads)
        nn.init.zeros_(self.rel_bias.weight)

    def forward(self, seq_len: int) -> torch.Tensor:
        device = self.rel_bias.weight.device
        idx = torch.arange(seq_len, device=device)
        dist = (idx.unsqueeze(1) - idx.unsqueeze(0)).clamp(min=0)
        dist = dist.clamp(max=self.rel_bias.num_embeddings - 1)
        bias = self.rel_bias(dist)
        return bias.permute(2, 0, 1)

class CausalWindowAttnTCN(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=2, num_heads=4, dropout=0.1):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.window_size = max(2, 2 * kernel_size)
        head_dim = max(out_channels // num_heads, 1)
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scale = math.sqrt(head_dim)

        self.q_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(in_channels, num_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * head_dim, out_channels)
        self.rel_pe = RelativePositionalEncoding(num_heads)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(out_channels)
        self.gate = nn.Linear(out_channels, out_channels)

        self.residual_proj = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, N, T = x.shape
        x_bn = x.permute(0, 2, 3, 1).reshape(B * N, T, C)

        Q = self.q_proj(x_bn)
        K = self.k_proj(x_bn)
        V = self.v_proj(x_bn)

        H, D = self.num_heads, self.head_dim
        Q = Q.view(B * N, T, H, D).transpose(1, 2)
        K = K.view(B * N, T, H, D).transpose(1, 2)
        V = V.view(B * N, T, H, D).transpose(1, 2)

        attn = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        rel_bias = self.rel_pe(T)
        attn = attn + rel_bias.unsqueeze(0)

        mask = _causal_window_mask(T, self.window_size, x.device)
        attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).reshape(B * N, T, H * D)
        out = self.out_proj(out)
        out = out * torch.sigmoid(self.gate(out))
        out = self.norm(out)

        out = out.view(B, N, T, -1).permute(0, 3, 1, 2)
        res = self.residual_proj(x)
        out = out[:, :, :, 1:] + res[:, :, :, 1:]
        return out


# =============================================================================
# Dynamic + GCN
# =============================================================================
class DynamicAdaptiveAdj(nn.Module):
    def __init__(self, num_nodes, emb_dim=10, in_channels=32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=(1, 1), bias=True)

    def forward(self, x, nodevec1, nodevec2):
        z = x.mean(dim=-1, keepdim=True)
        z = self.proj(z).squeeze(-1)
        z = z.permute(0, 2, 1)
        nv1 = nodevec1.unsqueeze(0) + z
        nv2 = nodevec2.t().unsqueeze(0) + z
        logits = torch.bmm(nv1, nv2.permute(0, 2, 1))
        return F.softmax(F.relu(logits), dim=-1)


class nconv(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, x, A):
        return torch.einsum('ncvl,vw->ncwl', (x, A)).contiguous()

class linear(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Conv2d(c_in, c_out, kernel_size=(1, 1), bias=True)
    def forward(self, x): return self.mlp(x)

class gcn_patched(nn.Module):
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self._nconv_static = nconv()
        c_in_actual = (order * support_len + 1) * c_in
        self.mlp = linear(c_in_actual, c_out)
        self.dropout = dropout
        self.order = order

    def _nconv_dynamic(self, x, A):
        return torch.einsum('bcnl,bnm->bcml', x, A).contiguous()

    def forward(self, x, support):
        out = [x]
        for a in support:
            conv_fn = self._nconv_dynamic if a.dim() == 3 else self._nconv_static
            x1 = conv_fn(x, a)
            out.append(x1)
            for _ in range(2, self.order + 1):
                x2 = conv_fn(x1, a)
                out.append(x2)
                x1 = x2
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h


# =============================================================================
# FULL MODEL
# =============================================================================
class gwnet(nn.Module):
    def __init__(self, device, num_nodes, dropout=0.3, supports=None, gcn_bool=True, addaptadj=True, aptinit=None,
                 in_dim=2, out_dim=12, residual_channels=32, dilation_channels=32,
                 skip_channels=256, end_channels=512, kernel_size=2, blocks=4, layers=2,
                 emb_dim=4, topk=10):
        
        super().__init__()
        self.dropout = dropout
        self.blocks = blocks
        self.layers = layers
        self.gcn_bool = gcn_bool
        self.addaptadj = addaptadj
        self.supports = supports
        self.topk = topk

        self.supports_len = 0 if supports is None else len(supports)
        if gcn_bool and addaptadj:
            if supports is None: self.supports = []
            if aptinit is None:
                self.nodevec1 = nn.Parameter(torch.randn(num_nodes, emb_dim).to(device), requires_grad=True)
                self.nodevec2 = nn.Parameter(torch.randn(emb_dim, num_nodes).to(device), requires_grad=True)
            else:
                m, p, n = torch.svd(aptinit)
                initemb1 = torch.mm(m[:, :emb_dim], torch.diag(p[:emb_dim] ** 0.5))
                initemb2 = torch.mm(torch.diag(p[:emb_dim] ** 0.5), n[:, :emb_dim].t())
                self.nodevec1 = nn.Parameter(initemb1.to(device), requires_grad=True)
                self.nodevec2 = nn.Parameter(initemb2.to(device), requires_grad=True)
            self.supports_len += 1

        self.dyn_adj = DynamicAdaptiveAdj(num_nodes, emb_dim, dilation_channels) if gcn_bool and addaptadj else None

        self.skip_attentions = nn.ModuleList([
            LocalCausalSkipAttention(skip_channels, num_heads=4, window_size=16, dropout=dropout)
            for _ in range(blocks * layers)
        ])

        self.start_conv = nn.Conv2d(in_dim, residual_channels, kernel_size=(1, 1))
        self.tcn_layers = nn.ModuleList()
        self.gcn_layers = nn.ModuleList()
        self.skip_convs = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.bn = nn.ModuleList()

        receptive_field = 1
        for b in range(blocks):
            new_dilation = 1
            for _ in range(layers):
                self.tcn_layers.append(CausalWindowAttnTCN(residual_channels, dilation_channels, new_dilation, 4, dropout))
                self.skip_convs.append(nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1)))
                if gcn_bool:
                    self.gcn_layers.append(gcn_patched(dilation_channels, residual_channels, dropout, self.supports_len))
                else:
                    self.residual_convs.append(nn.Conv2d(dilation_channels, residual_channels, kernel_size=(1, 1)))
                self.bn.append(nn.BatchNorm2d(residual_channels))
                receptive_field += new_dilation
                new_dilation *= 2

        self.receptive_field = receptive_field
        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))
        self.end_conv_2 = nn.Conv2d(end_channels, out_dim, kernel_size=(1, 1))

    def _topk_sparse(self, adp):
        k = min(self.topk, adp.size(1))
        _, topk_idx = torch.topk(adp, k, dim=1)
        mask = torch.zeros_like(adp)
        mask.scatter_(1, topk_idx, 1.0)
        return adp * mask

    def forward(self, input):
        in_len = input.size(3)
        if in_len < self.receptive_field:
            input = F.pad(input, (self.receptive_field - in_len, 0, 0, 0))

        x = self.start_conv(input)
        static_supports = self.supports if self.supports is not None else []
        skip = 0
        gcn_idx = 0

        for layer_idx in range(self.blocks * self.layers):
            residual = x
            x_tcn = self.tcn_layers[layer_idx](x)

            # Local Causal Skip Attention
            s = self.skip_convs[layer_idx](x_tcn)
            s = self.skip_attentions[layer_idx](s)

            try:
                skip = skip[:, :, :, -s.size(3):]
            except:
                skip = 0
            skip = s + skip

            if self.gcn_bool and self.supports is not None:
                if self.addaptadj and self.dyn_adj is not None:
                    adp_dyn = self.dyn_adj(x_tcn, self.nodevec1, self.nodevec2)
                    current_supports = static_supports + [adp_dyn]
                    x = self.gcn_layers[gcn_idx](x_tcn, current_supports)
                else:
                    adp_full = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
                    adp_sparse = self._topk_sparse(adp_full)
                    x = self.gcn_layers[gcn_idx](x_tcn, static_supports + [adp_sparse])
                gcn_idx += 1
            else:
                x = self.residual_convs[layer_idx](x_tcn)

            x = x + residual[:, :, :, -x.size(3):]
            x = self.bn[layer_idx](x)

        x = F.relu(skip)
        x = F.relu(self.end_conv_1(x))
        x = self.end_conv_2(x)
        return x

In [ ]:
%%writefile /kaggle/working/Graph-WaveNet/train.py
import torch
import numpy as np
import argparse
import time
import util
import matplotlib.pyplot as plt
from engine import trainer

parser = argparse.ArgumentParser()
parser.add_argument('--device',type=str,default='cuda:3',help='')
parser.add_argument('--data',type=str,default='data/METR-LA',help='data path')
parser.add_argument('--adjdata',type=str,default='data/sensor_graph/adj_mx.pkl',help='adj data path')
parser.add_argument('--adjtype',type=str,default='doubletransition',help='adj type')
parser.add_argument('--gcn_bool',action='store_true',help='whether to add graph convolution layer')
parser.add_argument('--aptonly',action='store_true',help='whether only adaptive adj')
parser.add_argument('--addaptadj',action='store_true',help='whether add adaptive adj')
parser.add_argument('--randomadj',action='store_true',help='whether random initialize adaptive adj')
parser.add_argument('--seq_length',type=int,default=12,help='')
parser.add_argument('--nhid',type=int,default=32,help='')
parser.add_argument('--in_dim',type=int,default=2,help='inputs dimension')
parser.add_argument('--num_nodes',type=int,default=207,help='number of nodes')
parser.add_argument('--batch_size',type=int,default=64,help='batch size')
parser.add_argument('--learning_rate',type=float,default=0.001,help='learning rate')
parser.add_argument('--dropout',type=float,default=0.3,help='dropout rate')
parser.add_argument('--weight_decay',type=float,default=0.0001,help='weight decay rate')
parser.add_argument('--epochs',type=int,default=100,help='')
parser.add_argument('--print_every',type=int,default=50,help='')
#parser.add_argument('--seed',type=int,default=99,help='random seed')
parser.add_argument('--save',type=str,default='./garage/metr',help='save path')
parser.add_argument('--expid',type=int,default=1,help='experiment id')
parser.add_argument('--start_epoch', type=int, default=1)
parser.add_argument('--checkpoint', type=str, default=None)
args = parser.parse_args()


def main():
    #set seed
    #torch.manual_seed(args.seed)
    #np.random.seed(args.seed)
    #load data
    device = torch.device(args.device)
    sensor_ids, sensor_id_to_ind, adj_mx = util.load_adj(args.adjdata,args.adjtype)
    dataloader = util.load_dataset(args.data, args.batch_size, args.batch_size, args.batch_size)
    scaler = dataloader['scaler']
    supports = [torch.tensor(i).to(device) for i in adj_mx]

    print(args)

    if args.randomadj:
        adjinit = None
    else:
        adjinit = supports[0]

    if args.aptonly:
        supports = None

    engine = trainer(scaler, args.in_dim, args.seq_length, args.num_nodes, args.nhid, args.dropout,
                         args.learning_rate, args.weight_decay, device, supports, args.gcn_bool, args.addaptadj,
                         adjinit)

    # Load checkpoint SAU khi tạo engine
    if args.checkpoint:
        engine.model.load_state_dict(torch.load(args.checkpoint))
        print(f'Loaded checkpoint: {args.checkpoint}')

    print("start training...",flush=True)
    his_loss =[]
    val_time = []
    train_time = []
    for i in range(args.start_epoch, args.epochs + 1):
        #if i % 10 == 0:
            #lr = max(0.000002,args.learning_rate * (0.1 ** (i // 10)))
            #for g in engine.optimizer.param_groups:
                #g['lr'] = lr
        train_loss = []
        train_mape = []
        train_rmse = []
        t1 = time.time()
        dataloader['train_loader'].shuffle()
        for iter, (x, y) in enumerate(dataloader['train_loader'].get_iterator()):
            trainx = torch.Tensor(x).to(device)
            trainx= trainx.transpose(1, 3)
            trainy = torch.Tensor(y).to(device)
            trainy = trainy.transpose(1, 3)
            metrics = engine.train(trainx, trainy[:,0,:,:])
            train_loss.append(metrics[0])
            train_mape.append(metrics[1])
            train_rmse.append(metrics[2])
            if iter % args.print_every == 0 :
                log = 'Iter: {:03d}, Train Loss: {:.4f}, Train MAPE: {:.4f}, Train RMSE: {:.4f}'
                print(log.format(iter, train_loss[-1], train_mape[-1], train_rmse[-1]),flush=True)
        t2 = time.time()
        train_time.append(t2-t1)
        #validation
        valid_loss = []
        valid_mape = []
        valid_rmse = []

        s1 = time.time()
        for iter, (x, y) in enumerate(dataloader['val_loader'].get_iterator()):
            testx = torch.Tensor(x).to(device)
            testx = testx.transpose(1, 3)
            testy = torch.Tensor(y).to(device)
            testy = testy.transpose(1, 3)
            metrics = engine.eval(testx, testy[:,0,:,:])
            valid_loss.append(metrics[0])
            valid_mape.append(metrics[1])
            valid_rmse.append(metrics[2])
        s2 = time.time()
        log = 'Epoch: {:03d}, Inference Time: {:.4f} secs'
        print(log.format(i,(s2-s1)))
        val_time.append(s2-s1)
        mtrain_loss = np.mean(train_loss)
        mtrain_mape = np.mean(train_mape)
        mtrain_rmse = np.mean(train_rmse)

        mvalid_loss = np.mean(valid_loss)
        mvalid_mape = np.mean(valid_mape)
        mvalid_rmse = np.mean(valid_rmse)
        his_loss.append(mvalid_loss)

        log = 'Epoch: {:03d}, Train Loss: {:.4f}, Train MAPE: {:.4f}, Train RMSE: {:.4f}, Valid Loss: {:.4f}, Valid MAPE: {:.4f}, Valid RMSE: {:.4f}, Training Time: {:.4f}/epoch'
        print(log.format(i, mtrain_loss, mtrain_mape, mtrain_rmse, mvalid_loss, mvalid_mape, mvalid_rmse, (t2 - t1)),flush=True)
        # Save checkpoint mỗi epoch với tên epoch thật
        torch.save(engine.model.state_dict(), args.save+"_epoch_"+str(i)+"_"+str(round(mvalid_loss,2))+".pth")

    print("Average Training Time: {:.4f} secs/epoch".format(np.mean(train_time)))
    print("Average Inference Time: {:.4f} secs".format(np.mean(val_time)))

    #testing
    bestid = np.argmin(his_loss)
    best_epoch = args.start_epoch + bestid  # offset đúng epoch thật
    engine.model.load_state_dict(torch.load(args.save+"_epoch_"+str(best_epoch)+"_"+str(round(his_loss[bestid],2))+".pth"))

    outputs = []
    realy = torch.Tensor(dataloader['y_test']).to(device)
    realy = realy.transpose(1,3)[:,0,:,:]

    for iter, (x, y) in enumerate(dataloader['test_loader'].get_iterator()):
        testx = torch.Tensor(x).to(device)
        testx = testx.transpose(1,3)
        with torch.no_grad():
            preds = engine.model(testx).transpose(1,3)
        outputs.append(preds.squeeze())

    yhat = torch.cat(outputs,dim=0)
    yhat = yhat[:realy.size(0),...]

    print("Training finished")
    print("The valid loss on best model is", str(round(his_loss[bestid],4)))

    amae = []
    amape = []
    armse = []
    for i in range(12):
        pred = scaler.inverse_transform(yhat[:,:,i])
        real = realy[:,:,i]
        metrics = util.metric(pred,real)
        log = 'Evaluate best model on test data for horizon {:d}, Test MAE: {:.4f}, Test MAPE: {:.4f}, Test RMSE: {:.4f}'
        print(log.format(i+1, metrics[0], metrics[1], metrics[2]))
        amae.append(metrics[0])
        amape.append(metrics[1])
        armse.append(metrics[2])

    log = 'On average over 12 horizons, Test MAE: {:.4f}, Test MAPE: {:.4f}, Test RMSE: {:.4f}'
    print(log.format(np.mean(amae),np.mean(amape),np.mean(armse)))
    torch.save(engine.model.state_dict(), args.save+"_exp"+str(args.expid)+"_best_"+str(round(his_loss[bestid],2))+".pth")


if __name__ == "__main__":
    t1 = time.time()
    main()
    t2 = time.time()
    print("Total time spent: {:.4f}".format(t2-t1))

In [ ]:
%%writefile /kaggle/working/Graph-WaveNet/test.py
import util
import argparse
from model import *
import numpy as np
import pandas as pd

parser = argparse.ArgumentParser()
parser.add_argument('--device',type=str,default='cuda:0')
parser.add_argument('--data',type=str,default='data/METR-LA')
parser.add_argument('--adjdata',type=str,default='data/sensor_graph/adj_mx.pkl')
parser.add_argument('--adjtype',type=str,default='doubletransition')
parser.add_argument('--gcn_bool',action='store_true')
parser.add_argument('--aptonly',action='store_true')
parser.add_argument('--addaptadj',action='store_true')
parser.add_argument('--randomadj',action='store_true')
parser.add_argument('--seq_length',type=int,default=12)
parser.add_argument('--nhid',type=int,default=32)
parser.add_argument('--in_dim',type=int,default=2)
parser.add_argument('--num_nodes',type=int,default=207)
parser.add_argument('--batch_size',type=int,default=64)
parser.add_argument('--learning_rate',type=float,default=0.001)
parser.add_argument('--dropout',type=float,default=0.3)
parser.add_argument('--weight_decay',type=float,default=0.0001)
parser.add_argument('--checkpoint',type=str)
args = parser.parse_args()

def main():
    device = torch.device(args.device)

    _, _, adj_mx = util.load_adj(args.adjdata, args.adjtype)
    supports = [torch.tensor(i).to(device) for i in adj_mx]
    adjinit = None if args.randomadj else supports[0]
    if args.aptonly:
        supports = None

    model = gwnet(device, args.num_nodes, args.dropout, supports=supports,
                  gcn_bool=args.gcn_bool, addaptadj=args.addaptadj, aptinit=adjinit,
                  in_dim=args.in_dim, out_dim=args.seq_length,
                  residual_channels=args.nhid, dilation_channels=args.nhid,
                  skip_channels=args.nhid*8, end_channels=args.nhid*16)
    model.to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()
    print('model load successfully')

    dataloader = util.load_dataset(args.data, args.batch_size, args.batch_size, args.batch_size)
    scaler = dataloader['scaler']

    outputs = []
    realy = torch.Tensor(dataloader['y_test']).to(device)
    realy = realy.transpose(1,3)[:,0,:,:]  # (N, num_nodes, 12)

    for iter, (x, y) in enumerate(dataloader['test_loader'].get_iterator()):
        testx = torch.Tensor(x).to(device)
        testx = testx.transpose(1,3)
        testx = nn.functional.pad(testx,(1,0,0,0))  # pad như engine
        with torch.no_grad():
            preds = model(testx)          # (batch, 12, num_nodes, 5)
            preds = preds.transpose(1,3)  # (batch, 5, num_nodes, 12)
            preds = preds[:,0,:,:]        # (batch, num_nodes, 12)
        outputs.append(preds)

    yhat = torch.cat(outputs, dim=0)
    yhat = yhat[:realy.size(0),...]

    amae, amape, armse = [], [], []
    for i in range(12):
        pred = scaler.inverse_transform(yhat[:,:,i])
        real = realy[:,:,i]
        metrics = util.metric(pred, real)
        print('Horizon {:d}, MAE: {:.4f}, MAPE: {:.4f}, RMSE: {:.4f}'.format(
              i+1, metrics[0], metrics[1], metrics[2]))
        amae.append(metrics[0])
        amape.append(metrics[1])
        armse.append(metrics[2])

    print('Average | MAE: {:.4f}, MAPE: {:.4f}, RMSE: {:.4f}'.format(
          np.mean(amae), np.mean(amape), np.mean(armse)))

if __name__ == "__main__":
    main()

In [ ]:
!pip install -r requirements.txt

In [ ]:
!rm -rf data/METR-LA data/PEMS-BAY

!python generate_training_data.py \
    --output_dir=data/METR-LA \
    --traffic_df_filename=/kaggle/input/datasets/annnnguyen/metr-la-dataset/METR-LA.h5

!python generate_training_data.py \
    --output_dir=data/PEMS-BAY \
    --traffic_df_filename=/kaggle/input/datasets/scchuy/pemsbay/pems-bay.h5

In [ ]:
# Tạo thư mục lưu checkpoint trước
!mkdir -p garage

In [ ]:
!python train.py \
    --device cuda:0 \
    --data data/METR-LA \
    --adjdata data/sensor_graph/adj_mx.pkl \
    --gcn_bool \
    --addaptadj \
    --num_nodes 207 \
    --start_epoch 86 \
    --epochs 100 \
    --save ./garage/metr_full_model \
    --checkpoint /kaggle/input/datasets/kghangco/pyspark-ds200/metr_full_model_epoch_85_2.85.pth

In [ ]:
!ls -lh data/METR-LA

In [ ]:
 !python test.py \
     --device cuda:0 \
     --data data/METR-LA \
     --adjdata data/sensor_graph/adj_mx.pkl \
     --adjtype doubletransition \
     --gcn_bool \
     --addaptadj \
     --num_nodes 207 \
     --checkpoint /kaggle/input/datasets/kghangco/pyspark-ds200/metr_full_model_epoch_57_2.82.pth

In [ ]:
!cat /kaggle/working/Graph-WaveNet/engine.py

In [ ]:
# !python test.py \
#     --device cuda:0 \
#     --data data/METR-LA \
#     --adjdata data/sensor_graph/adj_mx.pkl \
#     --adjtype doubletransition \
#     --gcn_bool \
#     --addaptadj \
#     --num_nodes 207 \
#     --checkpoint /kaggle/input/graohwavenet/_epoch_53_2.79.pth